In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import os
import yaml
from pathlib import Path
from dask.distributed import Client
import dask.dataframe as dd
import networkx as nx
import sys
import pandas as pd


import ipycytoscape
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import intervals
import pygtrie
import seaborn as sns


/usr/workspace/pandey2/DFT/envDFT/lib/python3.9/site-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [3]:

use_local=False
if not use_local:
    with open(f'/g/g91/pandey2/.dftracer/configuration.yaml', 'r') as file:
        dlp_yaml = yaml.safe_load(file)
        app_root = dlp_yaml["app"]
else:
    app_root = str(Path(os.getcwd()).parent.parent)
sys.path.insert(0, app_root)

import dfanalyzer
print(dfanalyzer.__file__)
from dfanalyzer.main import DFAnalyzer,get_dft_configuration,update_dft_configuration,setup_logging,setup_dask_cluster, reset_dask_cluster, get_dft_configuration
from dfanalyzer.graph_visualization.cytoscape import GraphFunctions, CytoGraph
from dfanalyzer.graph import DFGrepInterference, DFGrepWorkflow 

if not use_local:
    dask_run_dir = os.path.join(app_root, "dfanalyzer", "dask", "run_dir")
    with open (os.path.join(dask_run_dir, f"scheduler_{os.getenv('USER')}.json"), "r") as f:
        dask_scheduler = json.load(f)["address"]
else:
    dask_scheduler = None

# App Name
app_name = "montage" 

condition_fn = None #

if app_name == "mummi":
    filename = "/usr/workspace/iopp/graph-io/dlp_logs/mummi-32-node/*pfw.gz"

elif app_name == "montage":
    filename ="/usr/workspace/iopp/graph-io/dlp_logs/montage_16_48ppn/montage*.pfw"
    cp_dir = "/p/lustre2/pandey2/cp_dir/montage"

elif app_name == "montage2m2d":
    filename = "/usr/workspace/iopp/graph-io/dlp_logs/montage-2mass-2-degree/*.pfw.gz"
    cp_dir = "/p/lustre2/pandey2/cp_dir/montage"

elif app_name == "montage2m7d":
    filename = "/usr/workspace/iopp/graph-io/dlp_logs/montage-2mass-7-degree/*.pfw.gz"
    cp_dir = "/p/lustre2/pandey2/cp_dir/montage"

elif app_name == "deepspeed":
    filename = "/usr/workspace/iopp/dlp_traces/deepspeed_8_4ppn/*.pfw.gz"

else:
    raise Exception("Unknown App name")


# Configuration 4 update log file dlp -> df
conf = update_dft_configuration(dask_scheduler=dask_scheduler, verbose=True, 
                                log_file=f"./dft_{os.getenv('USER')}.log", rebuild_index=False, time_approximate=False, 
                                host_pattern=r'lassen(\d+)', time_granularity=30e6, skip_hostname=True, conditions=condition_fn)
conf = get_dft_configuration()


# Setup
setup_logging()
setup_dask_cluster()
reset_dask_cluster()

/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/__init__.py


[INFO] [09:33:52] Initialized Client with 1872 workers and link http://134.9.71.28:8787/status [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:673]
[INFO] [09:34:22] Restarting all workers [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:665]


In [4]:
def all_mount_points():
    with open("/proc/mounts", "r") as file:
        mount_points = [line.split()[1] for line in file]
    with open("/usr/workspace/pandey2/lassen_mounts", "r") as file:
        mount_p = [line.split()[1] for line in file]
    return mount_points+mount_p

mount_points = all_mount_points()
trie = pygtrie.StringTrie(zip(mount_points, [True] * len(mount_points)))


In [5]:
def montage_cols_function(json_object, current_dict, time_approximate,condition_fn,load_data):
    d = {}
    def find_mount_point(path,trie):
        mount_point = trie.longest_prefix(path)
        if mount_point:
            return mount_point.key
        return '/'.join(path.split('/', 3)[:3])
    
    if "args" in json_object:
        if "fname" in json_object["args"]:
            d["filename"] = str(json_object["args"]["fname"])   
            d["mount_point"] = find_mount_point(trie=load_data["mount_point"],path=str(json_object["args"]["fname"]))    
    return d

load_cols_montage = {'filename':"string[pyarrow]",'mount_point':"string[pyarrow]"}


In [6]:
analyzer_montage = DFAnalyzer(filename,load_fn=montage_cols_function, load_cols=load_cols_montage, load_data={"mount_point":trie})

[INFO] [09:34:23] Created index for 0 files [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:371]
[INFO] [09:34:23] Total size of all files are <dask.bag.core.Item object at 0x1554aaec16d0> bytes [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:373]
[INFO] [09:34:46] Loaded events [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:431]
[INFO] [09:34:46] Loaded plots with slope threshold: 45 [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:437]


In [13]:
analyzer_montage.events.compute()

,name,cat,pid,tid,ts,te,dur,tinterval,trange,hostname,compute_time,io_time,app_io_time,total_time,filename,phase,size,mount_point
0,readlink,POSIX,826362,1652724,0,13,13,<NA>,0,corona235,<NA>,13,<NA>,13,/proc/self/exe,2,<NA>,/proc
1,access,POSIX,826362,1652724,109,157,48,<NA>,0,corona235,<NA>,48,<NA>,48,/usr/libexec/flux/cmd/flux-run.py,2,<NA>,/usr/libexec
2,readlink,POSIX,826362,1652724,178,184,6,<NA>,0,corona235,<NA>,6,<NA>,6,/proc/self/exe,2,<NA>,/proc
0,open,POSIX,826362,1652724,480387,480399,12,<NA>,0,corona235,<NA>,12,<NA>,12,/dev/urandom,2,<NA>,/dev
1,read,POSIX,826362,1652724,480515,480523,8,<NA>,0,corona235,<NA>,8,<NA>,8,/dev/urandom,2,4,/dev
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2890,read,POSIX,826362,1652724,256505,256508,3,<NA>,0,corona235,<NA>,3,<NA>,3,/dev/urandom,2,4,/dev
2891,close,POSIX,826362,1652724,256517,256520,3,<NA>,0,corona235,<NA>,3,<NA>,3,/dev/urandom,2,<NA>,/dev
2892,fopen,STDIO,826362,1652724,256769,256777,8,<NA>,0,corona235,<NA>,<NA>,<NA>,0,/etc/flux/security/conf.d/sign.toml,0,<NA>,/etc/flux
2893,fread,STDIO,826362,1652724,256792,256797,5,<NA>,0,corona235,<NA>,<NA>,<NA>,0,/etc/flux/security/conf.d/sign.toml,0,<NA>,/etc/flux


In [11]:
analyzer_montage.events.mount_point.unique().compute()

KilledWorker: Attempted to run task ('bag-from-delayed-file_to_blocks-filter-lambda-list-load_objects-to_dataframe-ef89c7d2270c8b1d4ae332574530e760', 494) on 4 different workers, but all those workers died while running it. The last worker that attempt to run the task was tcp://192.168.129.18:35159. Inspecting worker logs is often a good next step to diagnose what went wrong. For more information see https://distributed.dask.org/en/stable/killed.html.

In [7]:
analyzer_montage.events['id'] = analyzer_montage.events.index

In [22]:
# eventsDF = analyzer_montage.events[analyzer_montage.events['cat'] == "POSIX" ] # only posix events
# eventsDF.compute().mount_point.unique()

In [14]:

# eventsDF = analyzer_montage.events[analyzer_montage.events['cat'] == "POSIX" ] # only posix events

In [8]:
IFCalculator = DFGrepInterference(analyzer_montage.events, app_name=app_name, cp_dir=cp_dir, existing=False)

In [9]:
IFCalculator.get_degree()
IFCalculator.get_interference()
IFCalculator.get_interference_metadata()

KeyboardInterrupt: 

In [10]:
IFCalculator.write_checkpoint("inter", cp_dir=cp_dir)
IFCalculator.write_checkpoint("inter_metadata", cp_dir=cp_dir)